# 🛡️ Detección de Mensajes Spam y Phishing con Deep Learning (BiLSTM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andresbry2020/rnn/blob/main/notebooks/deep_learning_spam_lstm.ipynb)

**Autor:** Proyecto de Deep Learning para Datos Secuenciales  
**Arquitectura:** Red Neuronal Recurrente Bidireccional (BiLSTM) con Capa de Embeddings y Concat Mean-Max Pooling  
**Framework:** PyTorch  
**Dataset:** SMS Spam Collection Dataset (UCI Machine Learning Repository)  

---

## 📌 1. Selección y Justificación del Dataset y la Arquitectura

### 1.1 El Problema: Filtrado de Spam en Mensajería Instantánea
El filtrado de mensajes spam y detección de phishing en comunicaciones móviles es un problema crítico de ciberseguridad. Los mensajes SMS presentan particularidades secuenciales que desafían los algoritmos clásicos (Naive Bayes, TF-IDF + Regresión Logística):
1. **Longitud variable y breve:** Mensajes cortos con alta densidad de palabras clave.
2. **Ofuscación deliberada:** Reemplazo de palabras por códigos ("WINNER!!", "claim now", "£900", números de teléfono).
3. **Dependencias contextuales bidireccionales:** Una palabra como *"free"* o *"win"* puede ser legítima o fraudulenta dependiendo de las palabras que la anteceden y preceden.

### 1.2 Justificación de la Red BiLSTM (Bidirectional Long Short-Term Memory)
- Las **RNN tradicionales** sufren del problema de desvanecimiento del gradiente (*vanishing gradient*), impidiendo aprender dependencias a mediano y largo plazo.
- Las **LSTM** solucionan esto mediante celdas de memoria y tres compuertas: *Forget Gate* (compuerta de olvido), *Input Gate* (compuerta de entrada) y *Output Gate* (compuerta de salida).
- Las **BiLSTM** procesan la secuencia en **ambas direcciones** (de izquierda a derecha y de derecha a izquierda), capturando el contexto previo y posterior de cada token de manera simultánea, resultando en una representación semántica superior para la clasificación de secuencias.

## ⚙️ 2. Configuración del Entorno y Reproducibilidad

In [ ]:
# En Google Colab, clonar el repositorio o instalar requerimientos si es necesario:
# !pip install -q torch pandas numpy matplotlib seaborn scikit-learn

import os
import re
import sys
import json
import random
from collections import Counter
from typing import List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Fijación de semillas para garantizar reproducibilidad determinística
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo activo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 📊 3. Carga de Datos y Análisis Exploratorio (EDA)

In [ ]:
# Descargar o cargar dataset
data_url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(data_url, sep='\t', header=None, names=['label', 'message'])

print(f"Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
display(df.head())

In [ ]:
# Análisis de distribución de clases
class_counts = df['label'].value_counts()
class_pcts = df['label'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    'Cantidad': class_counts,
    'Porcentaje (%)': class_pcts.round(2)
})
print("Distribución de Clases:")
display(summary_df)

# Longitud en caracteres y palabras
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

# Visualización exploratoria
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Gráfica de barras de clases
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Clases (Ham vs Spam)', fontweight='bold')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Número de Mensajes')
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)",
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold')

# 2. Distribución de longitud en caracteres
sns.histplot(data=df, x='char_length', hue='label', bins=40, kde=True, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Distribución de Longitud en Caracteres', fontweight='bold')
axes[1].set_xlabel('Número de Caracteres')
axes[1].set_xlim(0, 300)

# 3. Distribución de conteo de palabras
sns.boxplot(data=df, x='label', y='word_count', ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Conteo de Palabras por Clase', fontweight='bold')
axes[2].set_xlabel('Clase')
axes[2].set_ylabel('Número de Palabras')

plt.tight_layout()
plt.show()

## 🧹 4. Limpieza de Texto, Tokenización y Vocabulario

Se aplican las siguientes transformaciones secuenciales:
1. Normalización a minúsculas.
2. Reemplazo de hipervínculos por el token semántico `<url>`.
3. Reemplazo de símbolos de divisas (£, $, €, etc.) por `<money>`.
4. Reemplazo de números telefónicos o códigos largos por `<phone>`.
5. Reemplazo de dígitos numéricos por `<number>`.
6. Tokenización basada en expresiones regulares.
7. Construcción de vocabulario con soporte para padding (`<pad>` = 0) y palabras desconocidas fuera del vocabulario (`<unk>` = 1).

In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_IDX = 0
UNK_IDX = 1

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' <url> ', text)
    text = re.sub(r'\b\d{5,}\b', ' <phone> ', text)
    text = re.sub(r'[£$€¥₹]', ' <money> ', text)
    text = re.sub(r'\b\d+\b', ' <number> ', text)
    text = re.sub(r'[^\w\s<>]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text: str) -> List[str]:
    cleaned = clean_text(text)
    if not cleaned:
        return []
    return re.findall(r'<[a-z]+>|\b\w+\b', cleaned)

class Vocabulary:
    def __init__(self, max_size=6000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
        self.idx2word = {PAD_IDX: PAD_TOKEN, UNK_IDX: UNK_TOKEN}
        
    def __len__(self):
        return len(self.word2idx)
        
    def build_vocab(self, texts: List[str]):
        counter = Counter()
        for t in texts:
            counter.update(tokenize(t))
        sorted_words = [w for w, c in counter.most_common() if c >= self.min_freq]
        available = self.max_size - 2
        for idx, word in enumerate(sorted_words[:available], start=2):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            
    def text_to_indices(self, text: str, max_len=80) -> List[int]:
        tokens = tokenize(text)
        indices = [self.word2idx.get(t, UNK_IDX) for t in tokens]
        if len(indices) < max_len:
            indices = indices + [PAD_IDX] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        return indices

# Prueba de limpieza y tokenización
sample_spam = "WINNER!! Call 09061701461 to claim your £1,000 cash prize immediately! http://claim.com"
print("Texto original:", sample_spam)
print("Texto limpio:  ", clean_text(sample_spam))
print("Tokens:        ", tokenize(sample_spam))

## ✂️ 5. División Estratificada en Entrenamiento, Validación y Prueba

Dividimos los datos preservando rigurosamente la proporción de clases:
- **Entrenamiento (Train):** 70% (~3,900 muestras)
- **Validación (Val):** 15% (~836 muestras)
- **Prueba (Test):** 15% (~836 muestras)

> **Importante:** El vocabulario se construye **únicamente** a partir del subconjunto de entrenamiento para prevenir fuga de datos (*data leakage*).

In [ ]:
df['label_bin'] = df['label'].map({'ham': 0, 'spam': 1})

# División Train+Val y Test (15%)
df_train_val, df_test = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label_bin']
)

# División Train y Val (15% del total -> ~17.65% de train_val)
adj_val_size = 0.15 / (1.0 - 0.15)
df_train, df_val = train_test_split(
    df_train_val, test_size=adj_val_size, random_state=42, stratify=df_train_val['label_bin']
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Conjunto de Entrenamiento: {len(df_train)} muestras ({len(df_train)/len(df)*100:.1f}%) | Spam: {df_train['label_bin'].sum()}")
print(f"Conjunto de Validación:    {len(df_val)} muestras ({len(df_val)/len(df)*100:.1f}%) | Spam: {df_val['label_bin'].sum()}")
print(f"Conjunto de Prueba:        {len(df_test)} muestras ({len(df_test)/len(df)*100:.1f}%) | Spam: {df_test['label_bin'].sum()}")

# Construcción de vocabulario sobre train
vocab = Vocabulary(max_size=6000, min_freq=2)
vocab.build_vocab(df_train['message'].tolist())
print(f"\nTamaño final del vocabulario: {len(vocab)} tokens únicos")

In [ ]:
MAX_LEN = 80
BATCH_SIZE = 32

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=80):
        self.sequences = torch.tensor(
            [vocab.text_to_indices(t, max_len=max_len) for t in texts],
            dtype=torch.long
        )
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = TextDataset(df_train['message'], df_train['label_bin'].values, vocab, MAX_LEN)
val_dataset = TextDataset(df_val['message'], df_val['label_bin'].values, vocab, MAX_LEN)
test_dataset = TextDataset(df_test['message'], df_test['label_bin'].values, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 🧠 6. Construcción de la Red Neuronal BiLSTM

### Arquitectura Detallada:
1. **Capa Embedding:** Convierte índices enteros en vectores densos continuos de dimensión `embed_dim=128`.
2. **Dropout de Embeddings (0.3):** Previene sobreajuste al regularizar la representación de palabras.
3. **BiLSTM de 2 Capas:** Capas LSTM bidireccionales apiladas con `hidden_dim=128`. Procesa secuencias hacia adelante y hacia atrás.
4. **Mecanismo Global Mean + Max Pooling:** Extrae tanto la activación máxima (palabras clave detonantes) como el promedio temporal de las activaciones (contexto global).
5. **Cabezal de Clasificación Denso:** Capa lineal de reducción a 64 dimensiones + Batch Normalization + ReLU + Dropout (0.3) + Capa de proyección lineal a 1 logit de salida.

In [ ]:
class SpamBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dense_dim=64, dropout=0.3):
        super(SpamBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.embedding_dropout = nn.Dropout(p=dropout)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Concatena Mean Pooling y Max Pooling
        pooled_dim = hidden_dim * 2 * 2
        
        self.classifier = nn.Sequential(
            nn.Linear(pooled_dim, dense_dim),
            nn.BatchNorm1d(dense_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(dense_dim, 1)
        )
        
    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.embedding_dropout(embedded)
        
        lstm_out, _ = self.lstm(embedded)
        
        mean_pool = torch.mean(lstm_out, dim=1)
        max_pool, _ = torch.max(lstm_out, dim=1)
        pooled = torch.cat([mean_pool, max_pool], dim=1)
        
        logits = self.classifier(pooled)
        return logits
        
    def predict_proba(self, x):
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))

model = SpamBiLSTM(vocab_size=len(vocab), embed_dim=128, hidden_dim=128, num_layers=2, dense_dim=64, dropout=0.3).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal de parámetros entrenables: {total_params:,}")

## 🚀 7. Entrenamiento y Optimización

- **Función de Pérdida:** `BCEWithLogitsLoss` con `pos_weight` calibrado según la proporción de clases para penalizar errores en la clase minoritaria (spam).
- **Optimizador:** `AdamW` con `weight_decay=1e-4` para regularización L2.
- **Scheduler:** `ReduceLROnPlateau` que reduce el learning rate ante estancamiento.
- **Early Stopping & Checkpoint:** Guarda los pesos del modelo con mayor F1-Score en validación.

In [ ]:
num_neg = (df_train['label_bin'] == 0).sum()
num_pos = (df_train['label_bin'] == 1).sum()
pos_weight = torch.tensor([num_neg / (num_pos * 1.5)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

EPOCHS = 8

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def eval_loader(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_probs, all_targets = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            
            total_loss += loss.item() * x.size(0)
            all_probs.extend(probs.flatten())
            all_preds.extend(preds.flatten())
            all_targets.extend(y.cpu().numpy().flatten())
            
    all_targets = np.array(all_targets)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    return {
        "loss": total_loss / len(all_targets),
        "acc": accuracy_score(all_targets, all_preds),
        "f1": f1_score(all_targets, all_preds, zero_division=0),
        "prec": precision_score(all_targets, all_preds, zero_division=0),
        "rec": recall_score(all_targets, all_preds, zero_division=0),
        "auc": roc_auc_score(all_targets, all_probs),
        "preds": all_preds,
        "probs": all_probs,
        "targets": all_targets
    }

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_f1 = -1.0
best_model_state = None

print("Iniciando entrenamiento del modelo BiLSTM...\n")
for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_m = eval_loader(model, val_loader, criterion)
    scheduler.step(val_m["loss"])
    
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(val_m["loss"])
    history["val_acc"].append(val_m["acc"])
    history["val_f1"].append(val_m["f1"])
    
    if val_m["f1"] > best_f1:
        best_f1 = val_m["f1"]
        best_model_state = model.state_dict().copy()
        star = " [* MEJOR MODELO *]"
    else:
        star = ""
        
    print(f"Época {epoch:02d}/{EPOCHS:02d} | Train Loss: {t_loss:.4f} - Acc: {t_acc*100:.2f}% | Val Loss: {val_m['loss']:.4f} - Acc: {val_m['acc']*100:.2f}% - F1: {val_m['f1']:.4f}{star}")

# Cargar mejores pesos
model.load_state_dict(best_model_state)

## 📈 8. Gráficas del Proceso de Entrenamiento

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], 'o-', color='#1f77b4', label='Entrenamiento')
plt.plot(epochs_range, history["val_loss"], 's-', color='#ff7f0e', label='Validación')
plt.title('Curva de Pérdida (Loss) por Época', fontsize=12, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss (BCE)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], 'o-', color='#2ca02c', label='Entrenamiento')
plt.plot(epochs_range, history["val_acc"], 's-', color='#d62728', label='Validación')
plt.title('Curva de Exactitud (Accuracy) por Época', fontsize=12, fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

## 🎯 9. Evaluación en el Conjunto de Prueba Independiente (Test Set)

In [ ]:
test_m = eval_loader(model, test_loader, criterion)

print("="*60)
print("             MÉTRICAS EN CONJUNTO DE PRUEBA")
print("="*60)
print(f"  Exactitud (Accuracy):   {test_m['acc']*100:.2f}%")
print(f"  Precisión (Precision):  {test_m['prec']*100:.2f}%")
print(f"  Sensibilidad (Recall):  {test_m['rec']*100:.2f}%")
print(f"  F1-Score:               {test_m['f1']*100:.2f}%")
print(f"  Área bajo ROC (AUC):    {test_m['auc']:.4f}")
print("="*60)

print("\nReporte Detallado de Clasificación:")
print(classification_report(test_m["targets"], test_m["preds"], target_names=["Ham (Legítimo)", "Spam (Fraudulento)"], digits=4))

# Matriz de Confusión y Curva ROC
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(test_m["targets"], test_m["preds"])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
            annot_kws={"size": 14, "weight": "bold"})
axes[0].set_title('Matriz de Confusión en Prueba', fontweight='bold')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Etiqueta Real')

fpr, tpr, _ = roc_curve(test_m["targets"], test_m["probs"])
axes[1].plot(fpr, tpr, color='#9b59b6', lw=2, label=f'BiLSTM (AUC = {test_m["auc"]:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Curva ROC (Receiver Operating Characteristic)', fontweight='bold')
axes[1].set_xlabel('Tasa de Falsos Positivos (1 - Specificity)')
axes[1].set_ylabel('Tasa de Verdaderos Positivos (Recall)')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 📋 10. Mínimo 10 Predicciones de Prueba Detalladas

Se analiza una muestra diversa y equilibrada de 15 mensajes del test set:
- Texto original.
- Etiqueta real vs Etiqueta predicha.
- Probabilidad de Spam calculada por la red.
- Estado de acierto del modelo.

In [ ]:
df_test_eval = df_test.copy()
df_test_eval['predicted_label'] = np.where(test_m['preds'] == 1, 'spam', 'ham')
df_test_eval['spam_prob'] = test_m['probs']
df_test_eval['is_correct'] = df_test_eval['label'] == df_test_eval['predicted_label']

# Extraer 8 spam y 8 ham representativos
sample_spams = df_test_eval[df_test_eval['label'] == 'spam'].head(8)
sample_hams = df_test_eval[df_test_eval['label'] == 'ham'].head(8)
test_samples_df = pd.concat([sample_spams, sample_hams]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("="*92)
print(f"{'#':<3} | {'Mensaje SMS':<46} | {'Real':<6} | {'Pred':<6} | {'Prob Spam':<10} | {'Estado'}")
print("="*92)
for i, row in test_samples_df.iterrows():
    short = (row['message'][:43] + '...') if len(row['message']) > 43 else row['message']
    status = "✅ ACIERTO" if row['is_correct'] else "❌ FALLO"
    prob = f"{row['spam_prob']*100:.2f}%"
    print(f"{i+1:<3} | {short:<46} | {row['label']:<6} | {row['predicted_label']:<6} | {prob:<10} | {status}")
print("="*92)

## 💾 11. Guardado de Componentes y Despliegue con Streamlit

Guardamos los artefactos necesarios para que la aplicación Streamlit pueda operar de manera autónoma:
1. `best_bilstm_model.pt`: Pesos del modelo entrenado.
2. `vocab.json`: Mapeo de tokens a índices numéricos.
3. `model_config.json`: Hiperparámetros de la arquitectura.

Para ejecutar la aplicación interactiva Streamlit:
```bash
streamlit run app.py
```

In [ ]:
os.makedirs("saved_models", exist_ok=True)

# 1. Guardar modelo
torch.save(best_model_state, "saved_models/best_bilstm_model.pt")

# 2. Guardar vocabulario
vocab_data = {
    "max_size": vocab.max_size,
    "min_freq": vocab.min_freq,
    "word2idx": vocab.word2idx,
    "vocab_size": len(vocab)
}
with open("saved_models/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)

# 3. Guardar configuración
config = {
    "vocab_size": len(vocab),
    "embed_dim": 128,
    "hidden_dim": 128,
    "num_layers": 2,
    "bidirectional": True,
    "dense_dim": 64,
    "dropout": 0.3,
    "max_len": MAX_LEN
}
with open("saved_models/model_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("¡Artefactos guardados exitosamente en saved_models/!")